# Coursework: Self-supervised learning

In this coursework, you will explore the popular self-supervised contrastive learning approach [SimCLR]((https://arxiv.org/abs/2002.05709)).

You will be asked to implement some of the key components of SimCLR, including a suitable data augmentation strategy (for generating positive pairs), the SimCLR loss function, and the SimCLR training step. Additionally, you will be using transfer learning strategies for evaluating the performance of different pre-trained models for a downstream classification task.

The coursework is divided into three-parts:
- **Part A:** Implementation of a suitable dataset for contrastive model training;
- **Part B:** Implementation of the SimCLR loss and training step;
- **Part C:** Implementation of transfer learning strategies (linear probing and finetuning) for model evaluation.

**Important:** Read the text descriptions carefully and look out for hints and comments indicating a specific 'TASK'. Make sure to add sufficient documentation to your code.

**Submission:** You are asked to submit two versions of your notebook:
1. You should submit the raw notebook in `.ipynb` format with *all outputs cleared*. Please name your file `coursework.ipynb`.
2. Additionally, you will be asked to submit an exported version of your notebook in `.pdf` format, with *all outputs included*. We will primarily use this version for marking, but we will use the raw notebook to check for correct implementations. Please name this file `coursework_export.pdf`.

## Your details

Please add your details below. You can work in groups up to two.

Authors: **Piotr Blaszyk** & **Lizzie Williams**

DoC alias: **psb120** & **lsw120**

## Setup

In [ ]:
# On Google Colab uncomment the following line to install PyTorch Lightning and the MedMNIST dataset
! pip install lightning medmnist

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

from torch import linalg as LA
from torch.utils.data import DataLoader
from torchsummary import summary
from torchvision import models
from torchvision import transforms
from torchvision.transforms import v2
from pytorch_lightning import LightningModule, LightningDataModule, Trainer, seed_everything
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar
from torchmetrics.functional import auroc, accuracy
from PIL import Image
from medmnist.info import INFO
from medmnist.dataset import MedMNIST

## **Part A:** Implement a dataset suitable for contrastive learning.

We will be using the [MedMNIST Pneumonia](https://medmnist.com/) dataset, which is a medical imaging inspired dataset but with the characteristics of MNIST. This allows efficient experimentation due to the small image size. The dataset contains real chest X-ray images but downsampled to 28 x 28 pixels, with binary labels indicating the presence of [Pneumonia](https://www.nhs.uk/conditions/pneumonia/) (which is an inflammation of the lungs).

### **Task A-1:** Complete the dataset implementation.

You are asked to implement a dataset class `SimCLRPneumoniaMNISTDataset` suitable for training a self-supervised model with a contrastive objective. For each sample, your dataset class should return two 'views' of the corresponding image, forming the positive pairs for contrastive learning. It is up to you to design suitable augmentation pipeline for generating these views. Please provide a short description in plain language of what your data augmentation pipeline is meant to do.

To get you started, we have provided the skeleton of the dataset class in the cell below. Once you have implemented your dataset class, you are asked to run the provided visualisation code to visualise one batch of your training dataloader.

*Note:* You can use the same data augmentation pipeline for training, validation, and testing.

In [ ]:
class SimCLRPneumoniaMNISTDataset(MedMNIST):
    def __init__(self, split = 'train'):
        ''' Dataset class for PneumoniaMNIST.
        The provided init function will automatically download the necessary
        files at the first class initialistion.

        :param split: 'train', 'val' or 'test', select subset

        '''
        self.flag = "pneumoniamnist"
        self.size = 28
        self.size_flag = ""
        self.root = './data/coursework/'
        self.info = INFO[self.flag]
        self.download()

        npz_file = np.load(os.path.join(self.root, "pneumoniamnist.npz"))

        self.split = split

        # Load all the images
        assert self.split in ['train','val','test']

        self.imgs = npz_file[f'{self.split}_images']
        self.labels = npz_file[f'{self.split}_labels']

        # TASK: Define here your data augmentation pipeline
        # Add a short description in plain language.
        """
        I add augmentations for features that vary between images.
        I want the model to disregard these features.
        How much each features varies was determined via manual inspection
        of the 8 random image pairs from the visualisation cell below.
        Features that vary:
        * Zoom - hence RandomResizedCrop
        * Average pixel intensity - some images are brighter than others
          - hence ColorJitter
        * Rotation - hence RandomRotation
        Overall, the variance of each of the above features is rather small,
        hence the augmentations are also small.

        Augmentations are not applied to the features that I want the model
        to use. Those features remain the same across all images in each class.
        Some constant features (and related augmentations) include:
        * Vertical and horizontal flip - assume the patient always lies
          on their back with their head pointing up
        * Cutout - assume nothing is obscuring the view of the lungs
          in the scans
        * Sobel filtering - assume the scans are of high enough quality
          that more than just the contour is visible
        """
        self.augmentation_pipeline = v2.Compose([
          v2.RandomResizedCrop(size=28, scale=(0.5, 1.0), antialias=True),
          v2.ColorJitter(brightness=0.15,
                         contrast=None,
                         saturation=None,
                         hue=None),
          v2.RandomRotation(degrees=15),

          v2.ToImage(),
          v2.ToDtype(torch.float32, scale=True),
        ])

    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, index):
        # TASK: Fill in the blanks such that you return two tensors
        # of shape [1, 28, 28], img_view1 and img_view2, representing two augmented view of the images.
        img = self.imgs[index]
        img = img.reshape((1, 28, 28))
        img = torch.from_numpy(img)
        img_view1 = self.augmentation_pipeline(img)
        img_view2 = self.augmentation_pipeline(img)
        return img_view1, img_view2

We use a [LightningDataModule](https://lightning.ai/docs/pytorch/stable/data/datamodule.html) for handling your PneumoniaMNIST dataset. You do not need to make any modifications to the code below.

In [ ]:
class SimCLRPneumoniaMNISTDataModule(LightningDataModule):
    def __init__(self, batch_size: int = 8):
        super().__init__()
        self.batch_size = batch_size
        self.train_set = SimCLRPneumoniaMNISTDataset(split='train')
        self.val_set = SimCLRPneumoniaMNISTDataset(split='val')
        self.test_set = SimCLRPneumoniaMNISTDataset(split='test')

    def train_dataloader(self):
        return DataLoader(dataset=self.train_set, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(dataset=self.val_set, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(dataset=self.test_set, batch_size=self.batch_size, shuffle=False)

#### **Check** dataset implementation.

Run the below cell to visualise a batch of your training dataloader.

In [ ]:
# DO NOT MODIFY THIS CELL! IT IS FOR CHECKING THE IMPLEMENTATION ONLY.

# Initialise data module
datamodule = SimCLRPneumoniaMNISTDataModule()
# Get train dataloader
train_dataloader = datamodule.train_dataloader()
# Get first batch
batch = next(iter(train_dataloader))
# Visualise the images
view1, view2 = batch
f, ax = plt.subplots(2, 8, figsize=(12,4))
for i in range(8):
  ax[0,i].imshow(view1[i, 0], cmap='gray')
  ax[1,i].imshow(view2[i, 0], cmap='gray')
  ax[0,i].set_title('view 1')
  ax[1,i].set_title('view 2')
  ax[0, i].axis("off")
  ax[1, i].axis("off")

## **Part B:** Implement the SimCLR loss and training step.

In this part, we ask you to:
1. Implement the SimCLR loss function, as per the equation in the lecture notes (and the [original paper](https://arxiv.org/abs/2002.05709)).
2. Once you have implemented the loss, implement the training step function in the provided LightningModule.

### **Task B-1:** SimCLR loss function.

For the implementation of the SimCLR loss, you should follow the 'recipe' from the lecture slides. We provide a code skeleton to get you started. Fill in all the blanks.

*Hint:* In PyTorch, to compute scalar products (also called dot products) between many elements efficiently, note that for two batches of $d$-dimensional feature vectors $v1$ and $v2$ of size $[N, d]$ (with $N$ being the batch size) computing the matrix multiplication `torch.mm(v1, v2.t())` returns a matrix $S$ of size $[N, N]$ where each element $S[i, j]$ is the scalar product of $v1_i$ and $v2_j$.

In [ ]:
def simclr_loss(embedding_view1, embedding_view2, tau = 1.0):
  '''
  This funtion implements the SimCLR loss function as described in the original paper.
  See lecture notes for formulas.

  It takes as input the embeddings from both views and returns the loss value for that batch.
  Args:
    embedding_view1: torch tensor of shape [batch_size, embedding_dimension]
    embedding_view2: torch tensor of shape [batch_size, embedding_dimension]
  Returns:
    loss: torch.tensor of shape 1
  '''

  # Step 1: normalise the embeddings
  view1_norm = LA.vector_norm(embedding_view1, dim=1)
  view2_norm = LA.vector_norm(embedding_view2, dim=1)

  view1_norm = view1_norm.to(embedding_view1.device)
  view2_norm = view2_norm.to(embedding_view1.device)

  embedding_view1 = (embedding_view1.t() / view1_norm).t()
  embedding_view2 = (embedding_view2.t() / view2_norm).t()

  # Step 2: gather all embeddings into one big vector of size [2*N , feature_dim]
  z_all_views = torch.cat([embedding_view1, embedding_view2], dim=0)

  z_all_views = z_all_views.to(embedding_view1.device)

  # Step 3: compute all possible similarities, should be a matrix of size [2 * N, 2 * N]
  # all_similarities[i,j] will be the similarity between z_all_views[i] and z_all_views[j].
  # Use the hint.
  all_similarities = torch.mm(z_all_views, z_all_views.t())

  all_similarities = all_similarities.to(embedding_view1.device)

  # Step 4: Here we want to return a mask of size[2 * N, 2 * N] for which mask[i,j] = 1 if
  # z_all_views[i] and z_all_views[j] form a positive pair.
  # There should be exactly 2 * N non-zeros elements in this matrix.
  n = list(all_similarities.shape)[0] // 2
  mask = torch.zeros((2 * n, 2 * n), dtype=torch.int)
  mask = mask.fill_diagonal_(1)
  mask = torch.roll(mask, shifts=n, dims=0)

  mask = mask.to(embedding_view1.device)

  # Step 5: self-mask. For computing the denominator term in the loss function,
  # we need to sum over all possible similarities except the self-similarity.
  # Create a mask of shape [2*N, 2*N] that is 1 for all valid pairs and 0 for all self-pairs (i = j).
  self_mask = torch.ones((2 * n, 2 * n), dtype=torch.int)
  self_mask = self_mask.fill_diagonal_(0)

  self_mask = self_mask.to(embedding_view1.device)

  # Step 6: Computing all numerators for the loss function.
  # Should be vector of size [2 * N],
  # where element is exp(sim(i, j) / t) for each positive pair (i, j).
  # Re-use the computed quantities above.
  positive_modified = torch.exp(all_similarities / tau) * mask
  positive_modified = positive_modified.to(embedding_view1.device)

  numerators = torch.sum(positive_modified, dim=1)
  numerators = numerators.to(embedding_view1.device)

  # Step 7: Computing all denominators for the loss function.
  # Should be a vector of size [2 * N].
  # Where each element should be the sum of exp(sim(i,k)/tau) for all k != i.
  non_self_modified = torch.exp(all_similarities / tau) * self_mask
  non_self_modified = non_self_modified.to(embedding_view1.device)

  denominators = torch.sum(non_self_modified, dim=1)
  denominators = denominators.to(embedding_view1.device)

  # Step 8: Return the final loss values, using the previously computed numerators and denominators.
  all_loss = - torch.log(numerators / denominators)
  all_loss = all_loss.to(embedding_view1.device)

  batch_loss = torch.mean(all_loss)

  return batch_loss

#### **Check** SimCLR loss function.

To check your implementation, please run the following tests. Note that we will also use other tests on different inputs to test your code.

In [ ]:
# DO NOT MODIFY THIS CELL! IT IS FOR CHECKING THE IMPLEMENTATION ONLY.

seed_everything(33)

expected_results = [torch.tensor(1.7518), torch.tensor(1.6376), torch.tensor(4.194),  torch.tensor(4.1754)]
for i, (N, feature_dim) in enumerate(zip([3, 3, 33, 33], [5, 125, 5, 125])):
  embedding_view1 = torch.rand((N, feature_dim))
  embedding_view2 = torch.rand((N, feature_dim))
  loss = simclr_loss(embedding_view1.clone(), embedding_view2.clone(), tau=0.5)
  print(f"Expected loss: {expected_results[i]}, Computed loss: {loss}")
  assert torch.isclose(loss, expected_results[i], rtol=1e-3)
print("Passed all tests successfully !")

### **Task B-2:** SimCLR training step.

In this next task you are asked to complete the blanks in the provided [LightningModule](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html).

We provide the implementation of an image encoder (the CNN backbone that will act as feature extractor). No changes are needed for this part.

In [ ]:
class ImageEncoder(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = models.resnet50(weights=None)
        del self.net.fc
        self.net.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.net.conv1(x)
        x = self.net.bn1(x)
        x = self.net.relu(x)
        x0 = self.net.maxpool(x)
        x1 = self.net.layer1(x0)
        x2 = self.net.layer2(x1)
        x3 = self.net.layer3(x2)
        x4 = self.net.layer4(x3)
        x4 = self.net.avgpool(x4)
        x4 = torch.flatten(x4, 1)
        return x4

Next, you will need to complete the implementation of the SimCLR model. In order to make the training step work correctly, you will need to implement the `process_batch` function.

In [ ]:
class SimCLRModel(LightningModule):
    def __init__(self, learning_rate: float = 0.001):
        super().__init__()
        self.learning_rate = learning_rate

        self.encoder = ImageEncoder()

        self.projector = torch.nn.Sequential(
            torch.nn.Linear(2048, 1024),
            torch.nn.ReLU(),
            torch.nn.Linear(1024, 128),
        )

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

    def process_batch(self, batch):
        # TASK: Implement the process_batch function
        batch = [self.encoder(x) for x in batch]
        batch = [self.projector(x) for x in batch]
        loss = simclr_loss(batch[0], batch[1])
        return loss

    def training_step(self, batch, batch_idx):
        loss = self.process_batch(batch)
        self.log('train_loss', loss, prog_bar=True)
        if batch_idx == 0:
            grid = torchvision.utils.make_grid(torch.cat((batch[0][0:4, ...], batch[1][0:4, ...]), dim=0), nrow=4, normalize=True)
            self.logger.experiment.add_image('train_images', grid, self.global_step)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self.process_batch(batch)
        self.log('val_loss', loss, prog_bar=True)

#### **Check** SimCLR training step.

Here you can test that your code runs fine by training the model for 5 epochs using the cell below.

Report the training and validation loss at the end of 5 epochs.

In [ ]:
# DO NOT MODIFY THIS CELL! IT IS FOR CHECKING THE IMPLEMENTATION ONLY.

seed_everything(33, workers=True)

data = SimCLRPneumoniaMNISTDataModule(batch_size=32)

model = SimCLRModel()

trainer = Trainer(
    max_epochs=5,
    accelerator='auto',
    devices=1,
    logger=TensorBoardLogger(save_dir='./lightning_logs/coursework/', name='simclr'),
    callbacks=[ModelCheckpoint(monitor='val_loss', mode='min'), TQDMProgressBar(refresh_rate=10)],
)
trainer.fit(model=model, datamodule=data)

### SimCLR training results

train_loss = 3.4913

val_loss = 3.4966

## **Part C:** Linear probing and model finetuning.

In this part, you are given two different image encoders that were pre-trained with different datasets and training strategies. The objective for this task is to assess the performance of these two encoders in a downstream classification task. To this end, you are asked to implement evaluation routines seen in the lecture: linear probing and model finetuning. The downstream task is the prediction of Pneumonia in the (small) chest X-ray images from the PneumoniaMNIST dataset.

This part can be broken down into the following tasks:
1. Adapt your PneunomiaMNIST dataset for the image classification task.
2. Implement a classification model with a linear layer attached to a pre-trained image encoder.
3. For both pre-trained encoders:
    - a) Train the classifier on top of the frozen encoder (linear probing)
    - b) Finetune the entire model (including the encoder).
4. Evaluate all models on the test set, and provide a brief summary (no more than 300 words) with an analysis of your findings.

### **Task C-1:** Adapt your PneunomiaMNIST dataset for the image classification task.

We can base our implementation largely on the `SimCLRPneumoniaMNISTDataset` and adapt it to make it suitable for image classification. Think about a suitable data augmentation pipeline. Check previous tutorials for inspiration.

In [ ]:
class PneumoniaMNISTDataset(MedMNIST):
    def __init__(self, split = 'train', augmentation: bool = False):
        ''' Dataset class for Pneumonia MNST.
        The provided init function will automatically download the necessary
        files at the first class initialistion.

        :param split: 'train', 'val' or 'test', select subset

        '''
        self.flag = "pneumoniamnist"
        self.size = 28
        self.size_flag = ""
        self.root = './data/coursework/'
        self.info = INFO[self.flag]
        self.download()

        npz_file = np.load(os.path.join(self.root, "pneumoniamnist.npz"))

        self.split = split

        # Load all the images
        assert self.split in ['train','val','test']

        self.imgs = npz_file[f'{self.split}_images']
        self.labels = npz_file[f'{self.split}_labels']

        self.do_augment = augmentation

        # TASK: Define here your data augmentation pipeline suitable for classification.
        # Check previous tutorials for inspiration.
        self.augmentation_pipeline = v2.Compose([
          v2.RandomResizedCrop(size=28, scale=(0.5, 1.0), antialias=True),
          v2.ColorJitter(brightness=0.15, contrast=None, saturation=None, hue=None),
          v2.RandomRotation(degrees=15),

          v2.ToImage(),
          v2.ToDtype(torch.float32, scale=True),
        ])

        self.non_augmentation_pipeline = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])

    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, index):
        # TASK: Implement the __getitem__ function to return the image and its class label.
        label = self.labels[index]
        img = self.imgs[index]
        img = img.reshape((1, 28, 28))
        img = torch.from_numpy(img)
        if self.do_augment:
          img = self.augmentation_pipeline(img)
        else:
          img = self.non_augmentation_pipeline(img)
        return img, label

Again, we use a [LightningDataModule](https://lightning.ai/docs/pytorch/stable/data/datamodule.html) for handling your PneumoniaMNIST dataset. No changes needed for this part.

In [ ]:
class PneumoniaMNISTDataModule(LightningDataModule):
    def __init__(self, batch_size: int = 32):
        super().__init__()
        self.batch_size = batch_size
        self.train_set = PneumoniaMNISTDataset(split='train', augmentation=True)
        self.val_set = PneumoniaMNISTDataset(split='val', augmentation=False)
        self.test_set = PneumoniaMNISTDataset(split='test', augmentation=False)

    def train_dataloader(self):
        return DataLoader(dataset=self.train_set, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(dataset=self.val_set, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(dataset=self.test_set, batch_size=self.batch_size, shuffle=False)

#### **Check** dataset implementation.

Run the below cell to visualise a batch of your training dataloader.

In [ ]:
# DO NOT MODIFY THIS CELL! IT IS FOR CHECKING THE IMPLEMENTATION ONLY.

# Initialise data module
datamodule = PneumoniaMNISTDataModule()
# Get train dataloader
train_dataloader = datamodule.train_dataloader()
# Get first batch
batch = next(iter(train_dataloader))
# Visualise the images
images, labels = batch
f, ax = plt.subplots(1, 8, figsize=(12,4))
for i in range(8):
  ax[i].imshow(images[i, 0], cmap='gray')
  ax[i].set_title('label: ' + str(labels[i].item()))
  ax[i].axis("off")

### **Task C-2:** Implement a classification model with a linear layer attached to a pre-trained image encoder.

We first download the weights of the two pre-trained image encoders. One of them has been trained with the self-supervised SimCLR objective on a large publicly available chest X-ray dataset (different from PneunomiaMNIST). The other encoder is a standard ImageNet backbone that has been trained with a supervised classification objective on the ImageNet dataset.

In [ ]:
! wget https://www.doc.ic.ac.uk/~bglocker/teaching/mli/coursework.zip
! unzip coursework.zip

We provide the function for loading the encoders. No changes needed here.

In [ ]:
def load_encoder_from_checkpoint(checkpoint_path):
  ckpt = torch.load(checkpoint_path, map_location='cpu')
  simclr_module = SimCLRModel()
  print(simclr_module.load_state_dict(state_dict=ckpt))
  return simclr_module.encoder.eval()

imagenet_model = './data/coursework/model_imagenet.ckpt'
chestxray_model = './data/coursework/model_chestxray.ckpt'

In [ ]:
model_names = [
    'imagenet',
    'chestxray'
]

def get_ckpt(model_name):
  return fr"./data/coursework/model_{model_name}.ckpt"

print(get_ckpt(model_names[0]))

Now, implement a classification model as a LightningModule for image classification using a pre-trained image encoder.

The model should have a flag in the init function `freeze_encoder` that if set to true freezes all the weights in the encoder (used for linear probing), and if set to false all weights are trainable (used for model finetuning).

*Hint:* Check out previous tutorials for inspiration on how to implement a classification model as LightningModule. For the coursework, we recommend using the Area Under the Receiver Operating Characteristic Curve (ROC-AUC) performance metric (instead of accuracy). ROC-AUC is measure of the overall discriminative power of a classification model. You can use the readily available implementation in [torchmetrics](https://lightning.ai/docs/torchmetrics/stable/classification/auroc.html#functional-interface). You should log the ROC-AUC similar to how we logged accuracy in previous tutorials.

In [ ]:
# TASK: Implement the ImageClassifier class
# Check previous tutorials for insipration how to implement an `ImageClassifier`
class ImageClassifier(LightningModule):
    def __init__(
        self,
        pretrained_encoder: torch.nn.Module,
        freeze_encoder: bool = True,
        output_dim: int = 2,
        learning_rate: float = 0.001
      ):

        super().__init__()

        self.freeze_encoder = freeze_encoder
        self.output_dim = output_dim
        self.learning_rate = learning_rate

        self.encoder = pretrained_encoder.train(mode=not freeze_encoder)

        self.decoder = torch.nn.Linear(2048, self.output_dim)

    def forward(self, x):
        if self.freeze_encoder:
          with torch.no_grad():
            x = self.encoder(x)
        else:
          x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.decoder(x), x

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

    def process_batch(self, batch):
        x, y = batch
        y = y.squeeze(dim=1)
        logits, xt = self(x)
        loss = F.cross_entropy(logits, y)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        acc = auroc(probs, y, task='multiclass', num_classes=self.output_dim)
        return loss, acc, xt

    def training_step(self, batch, batch_idx):
        loss, acc, xt = self.process_batch(batch)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_auroc', acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc, _ = self.process_batch(batch)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_auroc', acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        loss, acc, xt = self.process_batch(batch)
        self.log('test_loss', loss)
        self.log('test_auroc', acc)

### **Task C-3a:** Implement training and testing for linear probing.

Train two classification models using linear probing, one for each of the two provided image encoders. Evaluate on both the validation and test sets.

*Note:* Training for 25 epochs should be sufficient.

In [ ]:
def train_and_test(mode):

  seed_everything(33, workers=True)

  data = PneumoniaMNISTDataModule(batch_size=32)
  # TASK: Implement the linear probing training and testing routines.

  modes = [
      'finetuning',
      'linear-probing'
  ]

  assert(mode in modes)

  freeze_encoder = mode == modes[1]

  print()
  print('we freeze encoder' if freeze_encoder else 'we do NOT freeze encoder')

  for i in range(len(model_names)):

    model_name = model_names[i]

    print(model_name)
    print()

    model = ImageClassifier(
        pretrained_encoder=load_encoder_from_checkpoint(get_ckpt(model_name)),
        freeze_encoder=freeze_encoder,
    )

    trainer = Trainer(
        max_epochs=25,
        accelerator='auto',
        devices=1,
        logger=TensorBoardLogger(save_dir='./lightning_logs/coursework/', name=fr'image-classifier-{mode}-{model_name}'),
        callbacks=[ModelCheckpoint(monitor='val_loss', mode='min'), TQDMProgressBar(refresh_rate=10)],
    )
    trainer.fit(model=model, datamodule=data)

    trainer.validate(model=model, datamodule=data, ckpt_path=trainer.checkpoint_callback.best_model_path)

    trainer.test(model=model, datamodule=data, ckpt_path=trainer.checkpoint_callback.best_model_path)

    print()

print(models)

In [ ]:
train_and_test(mode='linear-probing')

### **Task C-3b:** Implement training and testing for model finetuning.

Repeat the experiments, but this time using model finetuning instead of linear probing. Evaluate on both the validation and test sets.

In [ ]:
train_and_test(mode='finetuning')

### **Task C-4:** Your evaluation report.

Provide a brief summary (no more than 300 words) with an analysis of your findings. Try explaining the observed performance.

AUC-ROC scores:

Linear probing:
1. ImageNet: 0.928
2. Chest x-ray: 0.865

Finetuning
1. ImageNet: 0.963
2. Chest x-ray: 0.967

All experiment runs yield a fairly high AUC-ROC score (> 85%). One reason behind that is that the chosen binary classification task is relatively easy, esp. when compared with ImageNet, which has 1,000 classes.

In the original SimCLR paper the supervised ResNet model consistently beats SimCLR in terms of accuracy assuming both models have the same number of parameters. Here, that’s also the case but only in linear probing - in finetuning the models have a similar performance. The superiority of ImageNet over SimCLR in the linear probing task is interesting given that ImageNet encoder has never trained on a chest x-ray. ImageNet has the advantage of being supervised and having been pre-trained on a larger and more diverse dataset than SimCLR, which could provide a more generalised feature representation. This allows it to perform well even when applied to this new domain, without needing to adjust its weights significantly.

Finetuning performed better than linear probing for both ImageNet and SimCLR which suggests that allowing pre-trained models to adjust their weights (finetuning) rather than keeping them frozen (linear probing) leads to better performance when adaptated to a new task. This is particularly evident in the case of ImageNet when despite being trained on a different data set than chest x-rays with finetuning it achieves high AUC-ROC scores.

In finetuning both models have a similar performance. This suggests that when both models are able to adjust their weights to the specific features of the chest x-ray images, the advantages of being pre-trained on a broader data-set are diminished.

The test loss values are consistent with AUC-ROC scores, i.e. the lower the loss value, the higher the AUC-ROC score.



## Logging

In [ ]:
%load_ext tensorboard
%tensorboard --logdir './lightning_logs/coursework/'